In [2]:
import sys
print(sys.version)


3.11.4 (tags/v3.11.4:d2340ef, Jun  7 2023, 05:45:37) [MSC v.1934 64 bit (AMD64)]


In [6]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import fitz
import pdfplumber
import os
import re
from io import StringIO
from urllib.parse import urljoin
from urllib.parse import urlparse

# Mumbwa Town Council Integrated Development Plan (IDP) Dataset

## 1. Introduction

This notebook documents the collection, extraction, cleaning, transformation,
and preprocessing of Integrated Development Plan (IDP) data published by
Mumbwa Town Council.

The purpose of this dataset is to provide structured information about
development priorities, planned projects, sectors, locations, funding
allocations, and other relevant development information contained in the
council's publicly available IDP documents.

## 2. Data Source

The primary data source is the official Mumbwa Town Council website:

https://www.mumbwacouncil.gov.zm/

IDP documents published by the council were identified and downloaded for
further extraction and analysis.

In [12]:
url = "https://www.mumbwacouncil.gov.zm/"



In [9]:
%pip install --upgrade certifi requests urllib3

Note: you may need to restart the kernel to use updated packages.


In [14]:
url = "https://www.mumbwacouncil.gov.zm/"

response = requests.get(url, timeout=30, verify=False)

print("Status code:", response.status_code)
print("Website accessible:", response.ok)

c:\Users\Grace Musanga\AppData\Local\Programs\Python\Python311\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'www.mumbwacouncil.gov.zm'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Status code: 200
Website accessible: True


In [10]:
!pip install pdfplumber pypdf

In [22]:
tracker_url = "https://www.mumbwacouncil.gov.zm/?page_id=932"

tracker_response = requests.get(
    tracker_url,
    verify=False,
    timeout=100
)

tracker_soup = BeautifulSoup(tracker_response.text, "html.parser")

print("Status:", tracker_response.status_code)
print("Tables:", len(tracker_soup.find_all("table")))
print("Links:", len(tracker_soup.find_all("a")))

c:\Users\Grace Musanga\AppData\Local\Programs\Python\Python311\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'www.mumbwacouncil.gov.zm'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Status: 200
Tables: 0
Links: 114


In [23]:
tracker_links = []

for link in tracker_soup.find_all("a", href=True):
    text = link.get_text(" ", strip=True)
    href = urljoin(tracker_url, link["href"])

    tracker_links.append({
        "text": text,
        "url": href
    })

tracker_links_df = pd.DataFrame(tracker_links)

pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_rows", None)

display(tracker_links_df)

,text,url
0,,https://www.mumbwacouncil.gov.zm/
1,Home,https://www.mumbwacouncil.gov.zm/
2,About,https://www.mumbwacouncil.gov.zm/?page_id=932
3,Who we are,https://www.mumbwacouncil.gov.zm/?page_id=118
4,Senior Management,https://www.mumbwacouncil.gov.zm/?page_id=2877
5,Departments,https://www.mumbwacouncil.gov.zm/?page_id=770
6,Civic Leaders,https://www.mumbwacouncil.gov.zm/?page_id=2868
7,Council Chairman,https://www.mumbwacouncil.gov.zm/?page_id=2879
8,Nangoma,https://www.mumbwacouncil.gov.zm/?page_id=2873
9,Mumbwa Central,https://www.mumbwacouncil.gov.zm/?page_id=2871


In [25]:
IDP_url = "https://www.mumbwacouncil.gov.zm/?page_id=195"

response = requests.get(IDP_url, timeout=30, verify=False)

soup = BeautifulSoup(response.text, "html.parser")

print(soup.title.text)

c:\Users\Grace Musanga\AppData\Local\Programs\Python\Python311\Lib\site-packages\urllib3\connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'www.mumbwacouncil.gov.zm'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Publications – Mumbwa Town Council


In [ ]:
cdf_pdfs = tracker_links_df[
    tracker_links_df["url"].str.contains(
        r"\.pdf",
        case=False,
        na=False
    )
].copy()

print("CDF PDF documents found:", len(cdf_pdfs))

display(cdf_pdfs[["text", "url"]])

NameError: name 'soup' is not defined